In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import bitsandbytes as bnb
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
import evaluate
from peft import LoraConfig, get_peft_model, TaskType
from collections import Counter
from transformers import BitsAndBytesConfig
from collections import Counter

from dotenv import load_dotenv
import os
load_dotenv()
YOUR_HF_TOKEN = os.getenv("YOUR_HF_TOKEN")

/home/bistreamt/Desktop/master/research 4/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
            
        loss_fct = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # predictions = (logits > 0).astype(int)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
        "precision_macro": precision_score(labels, predictions, average="macro"),
        "recall_macro": recall_score(labels, predictions, average="macro")
    }

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
with open("data/supreme-court-data.json", 'r', encoding='utf-8') as f:
    data = json.load(f)

if isinstance(data, list):
    labels = [item["label"] for item in data if "label" in item]
    
    label_counts = Counter(labels)
    
    print("Number of samples per label:")
    for label, count in sorted(label_counts.items()):
        print(f"{label:12} : {count:4d} samples")
    
    print(f"\nTotal samples in Supreme Court Data: {len(data)}")

Number of samples per label:
admis        :  291 samples
inadmisibil  :  588 samples
nesoluționat :  105 samples
respins      :  416 samples

Total samples in Supreme Court Data: 1400


In [5]:
try:
    from huggingface_hub import login
    login(YOUR_HF_TOKEN)
except Exception as e:
    print("Hugging Face login skipped:", e)

In [6]:
from evaluate import load

clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")
# sacrebleu = evaluate.load("sacrebleu")
# bertscore = load("bertscore")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    results = clf_metrics.compute(predictions=predictions, references=labels)
    
    decoded_preds = [str(p) for p in predictions]
    decoded_labels = [str(l) for l in labels]
    
    results.update(rouge.compute(predictions=decoded_preds, references=decoded_labels))
    results.update(meteor.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels]))
    # results.update(sacrebleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels]))
    # results.update(bertscore.compute(predictions=decoded_preds, references=decoded_labels, lang="en"))
    
    return results

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/bistreamt/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/bistreamt/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/bistreamt/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Zero-SHOT

In [8]:
def prepare_dataset(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    label_map = {"respins": 0, "admis": 1}
    data = [item for item in data if item['label'] != 'inadmisibil']

    formatted_data = []
    for entry in data:
        desc = " ".join([entry.get(f"case_description_{i}", "") for i in range(1, 9)]).strip()
        
        # --- PROMPT ENGINEERING START ---
        
        just = " ".join([entry.get(f"justification_{i}", "") for i in range(1, 5)]).strip()
        instructional_prompt = (
            f"Decide verdictul acestui caz juridic:\n"
            f"Ai de ales între două răspunsuri: admis sau respins.\n"
            f"DESCRIERE: {desc}\n"
            f"JUSTIFICARE: {just}\n"
        )

        # --- PROMPT ENGINEERING END ---
        
        formatted_data.append({
            "text": instructional_prompt, 
            "label": label_map[entry["label"]]
        })
    
    train_data, val_data = train_test_split(
        formatted_data, 
        test_size=0.15, 
        stratify=[d["label"] for d in formatted_data], # Keep class ratios same
        random_state=42
    )
    
    return Dataset.from_list(train_data), Dataset.from_list(val_data)

train_raw, val_raw = prepare_dataset("data/supreme-court-data.json")

## Llama

In [9]:
model_name = "meta-llama/Llama-2-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)

device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float32, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = get_peft_model(model, peft_config)

training_args = TrainingArguments(
    output_dir="results/supreme-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.2,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Loading checkpoint shards: 100%|██████████| 2/2 [00:15<00:00,  7.66s/it]
Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-2-7b-hf and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum,Meteor
150,1.457300,1.205403,0.540984,0.461538,0.533333,0.406780,0.540984,0.000000,0.540984,0.540984,0.270492
300,1.402300,1.202857,0.532787,0.486486,0.519231,0.457627,0.532787,0.000000,0.532787,0.532787,0.266393
450,1.202900,1.515166,0.540984,0.243243,0.600000,0.152542,0.540984,0.000000,0.540984,0.540984,0.270492
600,1.666300,1.563803,0.532787,0.240000,0.562500,0.152542,0.532787,0.000000,0.532787,0.532787,0.266393
750,1.484200,1.543555,0.524590,0.256410,0.526316,0.169492,0.524590,0.000000,0.524590,0.524590,0.262295
900,1.519800,1.266022,0.540984,0.490909,0.529412,0.457627,0.540984,0.000000,0.540984,0.540984,0.270492
1050,1.134600,1.269736,0.573770,0.551724,0.561404,0.542373,0.565574,0.000000,0.573770,0.573770,0.286885
1200,0.999100,1.365213,0.549180,0.476190,0.543478,0.423729,0.549180,0.000000,0.549180,0.549180,0.274590
1350,1.245000,1.289859,0.532787,0.495575,0.518519,0.474576,0.532787,0.000000,0.532787,0.532787,0.266393


/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning:

TrainOutput(global_step=1380, training_loss=1.3433985309324403, metrics={'train_runtime': 969.2987, 'train_samples_per_second': 1.424, 'train_steps_per_second': 1.424, 'total_flos': 2.747309746028544e+16, 'train_loss': 1.3433985309324403, 'epoch': 2.0})

## RoLlama

In [10]:
model_name = "OpenLLM-Ro/RoLlama2-7b-Base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)

device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float32, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = get_peft_model(model, peft_config)

training_args = TrainingArguments(
    output_dir="results/supreme-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.2,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Loading checkpoint shards: 100%|██████████| 3/3 [00:15<00:00,  5.25s/it]
Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at OpenLLM-Ro/RoLlama2-7b-Base and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum,Meteor
150,1.590600,1.307037,0.483607,0.588235,0.478723,0.762712,0.483607,0.000000,0.483607,0.483607,0.241803
300,1.301800,1.330935,0.459016,0.571429,0.463158,0.745763,0.459016,0.000000,0.459016,0.459016,0.229508
450,1.037400,1.202273,0.532787,0.457143,0.521739,0.406780,0.532787,0.000000,0.532787,0.532787,0.266393
600,1.224900,1.354980,0.508197,0.333333,0.483871,0.254237,0.504098,0.000000,0.508197,0.508197,0.254098
750,1.320900,1.425894,0.500000,0.282353,0.461538,0.203390,0.500000,0.000000,0.500000,0.500000,0.250000
900,1.160000,1.280295,0.549180,0.421053,0.555556,0.338983,0.549180,0.000000,0.549180,0.549180,0.274590
1050,1.058800,1.242706,0.516393,0.404040,0.500000,0.338983,0.516393,0.000000,0.516393,0.516393,0.258197
1200,0.983500,1.330161,0.516393,0.378947,0.500000,0.305085,0.516393,0.000000,0.516393,0.516393,0.258197
1350,1.066800,1.180316,0.524590,0.462963,0.510204,0.423729,0.524590,0.000000,0.524590,0.524590,0.262295


/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning:

TrainOutput(global_step=1380, training_loss=1.190235392252604, metrics={'train_runtime': 968.3145, 'train_samples_per_second': 1.425, 'train_steps_per_second': 1.425, 'total_flos': 2.747309746028544e+16, 'train_loss': 1.190235392252604, 'epoch': 2.0})

## RoGemma

In [15]:
model_name = "OpenLLM-Ro/RoGemma-7b-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)

device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float32, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = get_peft_model(model, peft_config)

training_args = TrainingArguments(
    output_dir="results/supreme-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.2,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Loading checkpoint shards: 100%|██████████| 4/4 [00:16<00:00,  4.04s/it]
Some weights of GemmaForSequenceClassification were not initialized from the model checkpoint at OpenLLM-Ro/RoGemma-7b-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum,Meteor
150,2.373600,1.981240,0.483607,0.307692,0.437500,0.237288,0.483607,0.000000,0.483607,0.483607,0.241803
300,1.799400,1.587303,0.516393,0.415842,0.500000,0.355932,0.516393,0.000000,0.516393,0.516393,0.258197
450,2.093200,2.488846,0.500000,0.115942,0.400000,0.067797,0.500000,0.000000,0.500000,0.500000,0.250000
600,2.422300,1.918059,0.516393,0.191781,0.500000,0.118644,0.516393,0.000000,0.516393,0.516393,0.258197
750,1.671600,2.035437,0.540984,0.200000,0.636364,0.118644,0.540984,0.000000,0.540984,0.540984,0.270492
900,1.914200,2.568542,0.516393,0.658960,0.500000,0.966102,0.516393,0.000000,0.516393,0.516393,0.258197
1050,1.349000,1.290815,0.540984,0.461538,0.533333,0.406780,0.540984,0.000000,0.540984,0.540984,0.270492
1200,1.308400,1.290934,0.549180,0.485981,0.541667,0.440678,0.549180,0.000000,0.549180,0.549180,0.274590
1350,1.434600,1.283015,0.508197,0.523810,0.492537,0.559322,0.508197,0.000000,0.508197,0.508197,0.254098


/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning:

TrainOutput(global_step=1380, training_loss=1.8106433315553527, metrics={'train_runtime': 1101.8263, 'train_samples_per_second': 1.252, 'train_steps_per_second': 1.252, 'total_flos': 3.287400031715328e+16, 'train_loss': 1.8106433315553527, 'epoch': 2.0})

## RoMistral

In [14]:
model_name = "OpenLLM-Ro/RoMistral-7b-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)

device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float32, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = get_peft_model(model, peft_config)

training_args = TrainingArguments(
    output_dir="results/supreme-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.2,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Loading checkpoint shards: 100%|██████████| 3/3 [00:16<00:00,  5.36s/it]
Some weights of MistralForSequenceClassification were not initialized from the model checkpoint at OpenLLM-Ro/RoMistral-7b-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum,Meteor
150,3.112600,2.597043,0.475410,0.396226,0.446809,0.355932,0.475410,0.000000,0.475410,0.475410,0.237705
300,2.945200,2.494361,0.459016,0.440678,0.440678,0.440678,0.459016,0.000000,0.459016,0.459016,0.229508
450,2.678400,3.125215,0.475410,0.179487,0.368421,0.118644,0.475410,0.000000,0.475410,0.475410,0.237705
600,2.557900,2.455422,0.508197,0.347826,0.484848,0.271186,0.508197,0.000000,0.508197,0.508197,0.254098
750,2.509400,2.365656,0.483607,0.452174,0.464286,0.440678,0.483607,0.000000,0.483607,0.483607,0.241803
900,1.967500,2.855942,0.426230,0.527027,0.438202,0.661017,0.426230,0.000000,0.426230,0.426230,0.213115
1050,2.526300,2.276776,0.508197,0.482759,0.491228,0.474576,0.508197,0.000000,0.508197,0.508197,0.254098
1200,2.196200,2.316246,0.483607,0.503937,0.470588,0.542373,0.483607,0.000000,0.483607,0.483607,0.241803
1350,1.937400,2.177198,0.540984,0.450980,0.534884,0.389831,0.540984,0.000000,0.540984,0.540984,0.270492
1500,1.913300,2.516056,0.532787,0.359551,0.533333,0.271186,0.532787,0.000000,0.532787,0.532787,0.266393


/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning:

KeyboardInterrupt: 

## RoGPT2

In [13]:
model_name = "readerbench/RoGPT2-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Important for GPT2 padding
model.config.pad_token_id = tokenizer.pad_token_id

model = model.to(device)

print(f"Using device: {device}")

training_args = TrainingArguments(
    output_dir="results/supreme-bert-llm",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    warmup_ratio=0.1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    weight_decay=0.1,
    fp16=True,
    logging_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Map: 100%|██████████| 122/122 [00:00<00:00, 2167.85 examples/s]
Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at readerbench/RoGPT2-base and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum,Meteor
50,0.712900,0.668656,0.581967,0.556522,0.571429,0.542373,0.581967,0.000000,0.581967,0.581967,0.290984
100,0.649800,0.594917,0.688525,0.660714,0.698113,0.627119,0.692623,0.000000,0.688525,0.688525,0.344262
150,0.518800,0.515345,0.754098,0.722222,0.795918,0.661017,0.754098,0.000000,0.754098,0.754098,0.377049
200,0.307600,0.475522,0.811475,0.780952,0.891304,0.694915,0.811475,0.000000,0.811475,0.811475,0.405738
250,0.178300,0.508138,0.778689,0.787402,0.735294,0.847458,0.778689,0.000000,0.778689,0.778689,0.389344
300,0.089300,0.467456,0.803279,0.803279,0.777778,0.830508,0.803279,0.000000,0.803279,0.803279,0.401639
350,0.038500,0.463359,0.836066,0.824561,0.854545,0.796610,0.836066,0.000000,0.836066,0.836066,0.418033
400,0.021200,0.468861,0.836066,0.824561,0.854545,0.796610,0.836066,0.000000,0.836066,0.836066,0.418033


TrainOutput(global_step=440, training_loss=0.2873467514460737, metrics={'train_runtime': 301.4481, 'train_samples_per_second': 22.89, 'train_steps_per_second': 1.46, 'total_flos': 1802947579084800.0, 'train_loss': 0.2873467514460737, 'epoch': 10.0})

# Chain-of-Thought Prompting

In [8]:
def prepare_dataset(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    label_map = {"respins": 0, "admis": 1}
    data = [item for item in data if item['label'] != 'inadmisibil']

    formatted_data = []
    for entry in data:
        desc = " ".join([entry.get(f"case_description_{i}", "") for i in range(1, 9)]).strip()
        just = " ".join([entry.get(f"justification_{i}", "") for i in range(1, 5)]).strip()

        # --- PROMPT ENGINEERING START ---
        instructional_prompt = (
            f"Ești un expert legal și task-ul tău este de a analiza "
            f"un caz juridic și de a decide verdictul corect. "
            f"Analizează pas cu pas informațiile oferite înainte "
            f"de a formula răspunsul final. "
            f"Decide verdictul acestui caz juridic:\n"
            f"Ai de ales între două răspunsuri: admis sau respins.\n"
            f"Descriere caz: {desc}\n" 
            f"Justificare: {just}\n"
        )

        # --- PROMPT ENGINEERING END ---
        
        formatted_data.append({
            "text": instructional_prompt, 
            "label": label_map[entry["label"]]
        })

    train_data, val_data = train_test_split(
        formatted_data, 
        test_size=0.15, 
        stratify=[d["label"] for d in formatted_data], # Keep class ratios same
        random_state=42
    )
    
    return Dataset.from_list(train_data), Dataset.from_list(val_data)

train_raw, val_raw = prepare_dataset("data/supreme-court-data.json")

## Llama

In [9]:
model_name = "meta-llama/Llama-2-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)

device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float32, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = get_peft_model(model, peft_config)

training_args = TrainingArguments(
    output_dir="results/supreme-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.2,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.63s/it]
Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-2-7b-hf and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum,Meteor
150,1.398100,1.108480,0.540984,0.404255,0.542857,0.322034,0.540984,0.000000,0.540984,0.540984,0.270492
300,1.369100,1.044803,0.590164,0.583333,0.573770,0.593220,0.590164,0.000000,0.590164,0.590164,0.295082
450,1.192900,1.612297,0.565574,0.293333,0.687500,0.186441,0.565574,0.000000,0.565574,0.565574,0.282787
600,1.534500,1.592571,0.557377,0.270270,0.666667,0.169492,0.557377,0.000000,0.557377,0.557377,0.278689
750,1.505100,1.591550,0.524590,0.216216,0.533333,0.135593,0.524590,0.000000,0.524590,0.524590,0.262295
900,1.250100,1.250758,0.565574,0.442105,0.583333,0.355932,0.565574,0.000000,0.565574,0.565574,0.282787
1050,1.319000,1.349434,0.573770,0.469388,0.589744,0.389831,0.573770,0.000000,0.573770,0.573770,0.286885
1200,1.216600,1.345379,0.549180,0.444444,0.550000,0.372881,0.549180,0.000000,0.549180,0.549180,0.274590
1350,1.253000,1.289055,0.540984,0.490909,0.529412,0.457627,0.540984,0.000000,0.540984,0.540984,0.270492


/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning:

TrainOutput(global_step=1380, training_loss=1.3363943763401198, metrics={'train_runtime': 969.1524, 'train_samples_per_second': 1.424, 'train_steps_per_second': 1.424, 'total_flos': 2.747309746028544e+16, 'train_loss': 1.3363943763401198, 'epoch': 2.0})

## RoLlama

In [10]:
model_name = "OpenLLM-Ro/RoLlama2-7b-Base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)

device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float32, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = get_peft_model(model, peft_config)

training_args = TrainingArguments(
    output_dir="results/supreme-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.2,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Loading checkpoint shards: 100%|██████████| 3/3 [00:09<00:00,  3.26s/it]
Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at OpenLLM-Ro/RoLlama2-7b-Base and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum,Meteor
150,1.554400,1.236863,0.532787,0.636943,0.510204,0.847458,0.532787,0.000000,0.532787,0.532787,0.266393
300,1.346200,1.374364,0.540984,0.658537,0.514286,0.915254,0.540984,0.000000,0.540984,0.540984,0.270492
450,1.261100,0.931197,0.557377,0.550000,0.540984,0.559322,0.557377,0.000000,0.557377,0.557377,0.278689
600,1.247100,1.049900,0.532787,0.400000,0.527778,0.322034,0.532787,0.000000,0.532787,0.532787,0.266393
750,1.017700,1.090918,0.516393,0.337079,0.500000,0.254237,0.516393,0.000000,0.516393,0.516393,0.258197
900,1.153600,1.018548,0.549180,0.421053,0.555556,0.338983,0.549180,0.000000,0.549180,0.549180,0.274590
1050,1.149000,1.065312,0.540984,0.450980,0.534884,0.389831,0.540984,0.000000,0.540984,0.540984,0.270492
1200,1.105300,1.036695,0.532787,0.477064,0.520000,0.440678,0.532787,0.000000,0.532787,0.532787,0.266393
1350,0.952500,1.015271,0.581967,0.604651,0.557143,0.661017,0.581967,0.000000,0.581967,0.581967,0.290984


/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning:

TrainOutput(global_step=1380, training_loss=1.1889118236044178, metrics={'train_runtime': 972.8459, 'train_samples_per_second': 1.419, 'train_steps_per_second': 1.419, 'total_flos': 2.747309746028544e+16, 'train_loss': 1.1889118236044178, 'epoch': 2.0})

## RoGemma

In [9]:
model_name = "OpenLLM-Ro/RoGemma-7b-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)

device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float32, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = get_peft_model(model, peft_config)

training_args = TrainingArguments(
    output_dir="results/supreme-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.2,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Loading checkpoint shards: 100%|██████████| 4/4 [00:19<00:00,  4.89s/it]
Some weights of GemmaForSequenceClassification were not initialized from the model checkpoint at OpenLLM-Ro/RoGemma-7b-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum,Meteor
150,2.292500,1.513207,0.508197,0.625000,0.495050,0.847458,0.508197,0.000000,0.508197,0.508197,0.254098
300,1.946000,1.849694,0.516393,0.646707,0.500000,0.915254,0.516393,0.000000,0.516393,0.516393,0.258197
450,1.677700,2.986264,0.524590,0.033333,1.000000,0.016949,0.524590,0.000000,0.524590,0.524590,0.262295
600,2.482900,1.999132,0.549180,0.153846,0.833333,0.084746,0.549180,0.000000,0.549180,0.549180,0.274590
750,1.540700,1.905599,0.540984,0.151515,0.714286,0.084746,0.540984,0.000000,0.540984,0.540984,0.270492
900,1.946100,3.089680,0.475410,0.640449,0.478992,0.966102,0.475410,0.000000,0.475410,0.475410,0.237705
1050,1.941800,1.185816,0.540984,0.636364,0.515789,0.830508,0.540984,0.000000,0.540984,0.540984,0.270492
1200,0.889100,1.017860,0.540984,0.575758,0.520548,0.644068,0.540984,0.000000,0.540984,0.540984,0.270492
1350,1.302600,1.542901,0.524590,0.654762,0.504587,0.932203,0.524590,0.000000,0.524590,0.524590,0.262295


/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning:

TrainOutput(global_step=1380, training_loss=1.773185425910397, metrics={'train_runtime': 1099.0712, 'train_samples_per_second': 1.256, 'train_steps_per_second': 1.256, 'total_flos': 3.287400031715328e+16, 'train_loss': 1.773185425910397, 'epoch': 2.0})

## RoMistral

In [10]:
model_name = "OpenLLM-Ro/RoMistral-7b-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)

device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float32, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = get_peft_model(model, peft_config)

training_args = TrainingArguments(
    output_dir="results/supreme-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.2,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Loading checkpoint shards: 100%|██████████| 3/3 [00:15<00:00,  5.13s/it]
Some weights of MistralForSequenceClassification were not initialized from the model checkpoint at OpenLLM-Ro/RoMistral-7b-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum,Meteor
150,2.892800,2.523259,0.508197,0.491525,0.491525,0.491525,0.508197,0.000000,0.508197,0.508197,0.254098
300,3.401500,2.299279,0.500000,0.460177,0.481481,0.440678,0.500000,0.000000,0.500000,0.500000,0.250000
450,2.953700,3.546068,0.524590,0.236842,0.529412,0.152542,0.524590,0.000000,0.524590,0.524590,0.262295
600,3.711400,2.374640,0.500000,0.344086,0.470588,0.271186,0.500000,0.000000,0.508197,0.500000,0.250000
750,2.426000,2.331976,0.532787,0.400000,0.527778,0.322034,0.532787,0.000000,0.532787,0.532787,0.266393
900,2.149800,2.810293,0.524590,0.637500,0.504950,0.864407,0.524590,0.000000,0.524590,0.524590,0.262295
1050,2.348000,1.915761,0.606557,0.652174,0.569620,0.762712,0.606557,0.000000,0.606557,0.606557,0.303279
1200,1.908800,1.838566,0.598361,0.566372,0.592593,0.542373,0.598361,0.000000,0.598361,0.598361,0.299180
1350,2.609900,1.845348,0.622951,0.622951,0.603175,0.644068,0.622951,0.000000,0.622951,0.622951,0.311475


/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning:

TrainOutput(global_step=1380, training_loss=2.702646465577941, metrics={'train_runtime': 1055.3614, 'train_samples_per_second': 1.308, 'train_steps_per_second': 1.308, 'total_flos': 2.960350324457472e+16, 'train_loss': 2.702646465577941, 'epoch': 2.0})

## RoGPT2

In [11]:
model_name = "readerbench/RoGPT2-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Important for GPT2 padding
model.config.pad_token_id = tokenizer.pad_token_id

model = model.to(device)

print(f"Using device: {device}")

training_args = TrainingArguments(
    output_dir="results/supreme-bert-llm",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    warmup_ratio=0.1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    weight_decay=0.1,
    fp16=True,
    logging_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Map: 100%|██████████| 122/122 [00:00<00:00, 2538.90 examples/s]
Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at readerbench/RoGPT2-base and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum,Meteor
50,0.772600,0.776249,0.524590,0.516667,0.508197,0.525424,0.524590,0.000000,0.524590,0.524590,0.262295
100,0.684800,0.696302,0.622951,0.622951,0.603175,0.644068,0.622951,0.000000,0.622951,0.622951,0.311475
150,0.566900,0.634522,0.663934,0.637168,0.666667,0.610169,0.663934,0.000000,0.663934,0.663934,0.331967
200,0.379400,0.630624,0.704918,0.694915,0.694915,0.694915,0.713115,0.000000,0.704918,0.704918,0.352459
250,0.210900,0.776403,0.713115,0.744526,0.653846,0.864407,0.713115,0.000000,0.713115,0.713115,0.356557
300,0.084600,0.740626,0.729508,0.748092,0.680556,0.830508,0.729508,0.000000,0.729508,0.729508,0.364754
350,0.037300,0.796140,0.713115,0.724409,0.676471,0.779661,0.713115,0.000000,0.713115,0.713115,0.356557
400,0.015500,0.849803,0.729508,0.744186,0.685714,0.813559,0.729508,0.000000,0.729508,0.729508,0.364754


TrainOutput(global_step=440, training_loss=0.31388860954479736, metrics={'train_runtime': 165.6079, 'train_samples_per_second': 41.665, 'train_steps_per_second': 2.657, 'total_flos': 1802947579084800.0, 'train_loss': 0.31388860954479736, 'epoch': 10.0})